# Fallback Dataset: Sleep Quality → Stress

This notebook tests a fallback dataset for the missing bridge problem.

The current issue is that sleep efficiency does not yet connect to the output variables.

This fallback dataset does not use sleep efficiency directly, but it may give a stronger bridge:

Quality of Sleep → Stress Level

It also tests:

Physical Activity Level → Quality of Sleep

This could support a simpler full path:

exercise → sleep quality → stress

A model is only considered usable if it beats the baseline mean model on held-out test RMSE.

In [1]:
# ------------------------------------------------------------
# Fallback dataset test: Sleep Health & Lifestyle
# ------------------------------------------------------------

from pathlib import Path
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Install packages if needed
# ------------------------------------------------------------

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name

    try:
        __import__(import_name)
    except ImportError:
        print(f"{package_name} missing. Installing...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])


install_if_missing("pandas")
install_if_missing("numpy")
install_if_missing("matplotlib")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("kagglehub")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
GRAPH_DIR = PROJECT_DIR / "Graphs"

FALLBACK_DIR = DATA_DIR / "fallback_sleep_quality_stress"

for folder in [DATA_DIR, GRAPH_DIR, FALLBACK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:")
print(PROJECT_DIR)

print("\nFallback dataset folder:")
print(FALLBACK_DIR)


# ------------------------------------------------------------
# Automatically download dataset from KaggleHub
# ------------------------------------------------------------

dataset_slug = "uom190346a/sleep-health-and-lifestyle-dataset"

print("\nTrying to download dataset automatically from KaggleHub:")
print(dataset_slug)

try:
    downloaded_path = kagglehub.dataset_download(dataset_slug)
    downloaded_path = Path(downloaded_path)

    print("\nDataset downloaded to:")
    print(downloaded_path)

except Exception as error:
    downloaded_path = None

    print("\nAutomatic KaggleHub download failed.")
    print("Error:")
    print(error)

    print("\nThis usually means KaggleHub needs internet access or Kaggle authentication.")
    print("If this happens, tell me the error message and we will use a different automatic source or a local fallback.")


# ------------------------------------------------------------
# Find CSV file
# ------------------------------------------------------------

csv_candidates = []

if downloaded_path is not None and downloaded_path.exists():
    csv_candidates.extend(list(downloaded_path.rglob("*.csv")))

# Also search inside project folder, in case dataset already exists locally
csv_candidates.extend(list(PROJECT_DIR.rglob("*Sleep*Health*Lifestyle*.csv")))
csv_candidates.extend(list(PROJECT_DIR.rglob("*sleep*health*lifestyle*.csv")))

# Remove duplicates
csv_candidates = list(dict.fromkeys(csv_candidates))

if len(csv_candidates) == 0:
    raise FileNotFoundError(
        "Could not find the Sleep Health & Lifestyle CSV file. "
        "The automatic download may have failed."
    )

dataset_path = csv_candidates[0]

print("\nUsing dataset file:")
print(dataset_path)

sleep_lifestyle_raw = pd.read_csv(dataset_path)

print("\nDataset shape:")
print(sleep_lifestyle_raw.shape)

print("\nColumns:")
print(list(sleep_lifestyle_raw.columns))

display(sleep_lifestyle_raw.head())


# ------------------------------------------------------------
# Clean useful variables
# ------------------------------------------------------------

needed_columns = [
    "Sleep Duration",
    "Quality of Sleep",
    "Physical Activity Level",
    "Stress Level",
    "Daily Steps"
]

missing_columns = [
    column for column in needed_columns
    if column not in sleep_lifestyle_raw.columns
]

if len(missing_columns) > 0:
    raise ValueError(f"Missing expected columns: {missing_columns}")

model_data = sleep_lifestyle_raw[needed_columns].copy()

for column in needed_columns:
    model_data[column] = pd.to_numeric(
        model_data[column],
        errors="coerce"
    )

model_data = model_data.dropna().copy()

print("\nRows available after cleaning:")
print(len(model_data))

print("\nDescriptive statistics:")
display(model_data.describe().round(3))

print("\nCorrelations:")
display(model_data.corr().round(3))


# ------------------------------------------------------------
# Helper function: evaluate regression path
# ------------------------------------------------------------

def evaluate_path(data, predictors, target, path_name):
    clean_data = data[predictors + [target]].dropna().copy()

    X = clean_data[predictors]
    y = clean_data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42
    )

    # Baseline mean model
    baseline_model = DummyRegressor(strategy="mean")
    baseline_model.fit(X_train, y_train)

    baseline_predictions = baseline_model.predict(X_test)

    baseline_r2 = r2_score(y_test, baseline_predictions)
    baseline_mae = mean_absolute_error(y_test, baseline_predictions)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

    # Linear model
    linear_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    linear_model.fit(X_train, y_train)

    linear_train_predictions = linear_model.predict(X_train)
    linear_test_predictions = linear_model.predict(X_test)

    linear_train_r2 = r2_score(y_train, linear_train_predictions)
    linear_test_r2 = r2_score(y_test, linear_test_predictions)
    linear_mae = mean_absolute_error(y_test, linear_test_predictions)
    linear_rmse = np.sqrt(mean_squared_error(y_test, linear_test_predictions))

    # Random forest comparison
    random_forest_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=4,
        random_state=42
    )

    random_forest_model.fit(X_train, y_train)

    rf_train_predictions = random_forest_model.predict(X_train)
    rf_test_predictions = random_forest_model.predict(X_test)

    rf_train_r2 = r2_score(y_train, rf_train_predictions)
    rf_test_r2 = r2_score(y_test, rf_test_predictions)
    rf_mae = mean_absolute_error(y_test, rf_test_predictions)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_test_predictions))

    # Best model
    options = [
        ("baseline mean", baseline_rmse),
        ("linear regression", linear_rmse),
        ("random forest", rf_rmse)
    ]

    best_model, best_rmse = sorted(options, key=lambda item: item[1])[0]

    beats_baseline = best_rmse < baseline_rmse

    decision = (
        "use with caution"
        if beats_baseline and best_model != "baseline mean"
        else "do not use"
    )

    # Extract linear formulas
    scaler = linear_model.named_steps["scaler"]
    fitted_linear = linear_model.named_steps["model"]

    standardized_intercept = fitted_linear.intercept_
    standardized_coefficients = fitted_linear.coef_

    feature_means = scaler.mean_
    feature_sds = scaler.scale_

    raw_coefficients = standardized_coefficients / feature_sds
    raw_intercept = standardized_intercept - np.sum(
        standardized_coefficients * feature_means / feature_sds
    )

    standardized_parts = [
        f"{standardized_coefficients[i]:+.6f} * z({predictors[i]})"
        for i in range(len(predictors))
    ]

    raw_parts = [
        f"{raw_coefficients[i]:+.6f} * {predictors[i]}"
        for i in range(len(predictors))
    ]

    standardized_formula = (
        f"{target} = {standardized_intercept:.6f} "
        + " ".join(standardized_parts)
    )

    raw_formula = (
        f"{target} = {raw_intercept:.6f} "
        + " ".join(raw_parts)
    )

    result = {
        "path": path_name,
        "target": target,
        "predictors": ", ".join(predictors),
        "rows_used": len(clean_data),
        "baseline_r2": round(baseline_r2, 4),
        "baseline_mae": round(baseline_mae, 4),
        "baseline_rmse": round(baseline_rmse, 4),
        "linear_train_r2": round(linear_train_r2, 4),
        "linear_test_r2": round(linear_test_r2, 4),
        "linear_mae": round(linear_mae, 4),
        "linear_rmse": round(linear_rmse, 4),
        "rf_train_r2": round(rf_train_r2, 4),
        "rf_test_r2": round(rf_test_r2, 4),
        "rf_mae": round(rf_mae, 4),
        "rf_rmse": round(rf_rmse, 4),
        "best_model": best_model,
        "beats_baseline": "Yes" if beats_baseline else "No",
        "decision": decision,
        "raw_formula": raw_formula,
        "standardized_formula": standardized_formula
    }

    coefficient_rows = []

    coefficient_rows.append({
        "path": path_name,
        "term": "Intercept",
        "raw_coefficient": raw_intercept,
        "standardized_coefficient": standardized_intercept,
        "training_mean": "",
        "training_sd": ""
    })

    for i, predictor in enumerate(predictors):
        coefficient_rows.append({
            "path": path_name,
            "term": predictor,
            "raw_coefficient": raw_coefficients[i],
            "standardized_coefficient": standardized_coefficients[i],
            "training_mean": feature_means[i],
            "training_sd": feature_sds[i]
        })

    coefficient_table = pd.DataFrame(coefficient_rows)

    return result, coefficient_table


# ------------------------------------------------------------
# Test useful fallback paths
# ------------------------------------------------------------

model_jobs = [
    {
        "path_name": "Quality of Sleep → Stress Level",
        "predictors": ["Quality of Sleep"],
        "target": "Stress Level"
    },
    {
        "path_name": "Sleep Duration + Quality of Sleep → Stress Level",
        "predictors": ["Sleep Duration", "Quality of Sleep"],
        "target": "Stress Level"
    },
    {
        "path_name": "Physical Activity Level → Quality of Sleep",
        "predictors": ["Physical Activity Level"],
        "target": "Quality of Sleep"
    },
    {
        "path_name": "Physical Activity Level + Daily Steps → Quality of Sleep",
        "predictors": ["Physical Activity Level", "Daily Steps"],
        "target": "Quality of Sleep"
    },
    {
        "path_name": "Physical Activity Level → Stress Level",
        "predictors": ["Physical Activity Level"],
        "target": "Stress Level"
    },
    {
        "path_name": "Quality of Sleep + Physical Activity Level → Stress Level",
        "predictors": ["Quality of Sleep", "Physical Activity Level"],
        "target": "Stress Level"
    }
]

results = []
coefficient_tables = []

for job in model_jobs:
    result, coefficient_table = evaluate_path(
        data=model_data,
        predictors=job["predictors"],
        target=job["target"],
        path_name=job["path_name"]
    )

    results.append(result)
    coefficient_tables.append(coefficient_table)

fallback_results_table = pd.DataFrame(results)
fallback_coefficients_table = pd.concat(coefficient_tables, ignore_index=True)

print("\nFallback model results:")
display(fallback_results_table)

print("\nFallback coefficients:")
display(fallback_coefficients_table)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

results_path = PROJECT_DIR / "fallback_sleep_quality_stress_results.csv"
coefficients_path = PROJECT_DIR / "fallback_sleep_quality_stress_coefficients.csv"

fallback_results_table.to_csv(results_path, index=False)
fallback_coefficients_table.to_csv(coefficients_path, index=False)

print("\nSaved fallback model results to:")
print(results_path)

print("\nSaved fallback coefficients to:")
print(coefficients_path)


# ------------------------------------------------------------
# Print most important formulas clearly
# ------------------------------------------------------------

important_paths = [
    "Quality of Sleep → Stress Level",
    "Physical Activity Level → Quality of Sleep",
    "Quality of Sleep + Physical Activity Level → Stress Level"
]

print("\nIMPORTANT FALLBACK FORMULAS")
print("=" * 80)

for _, row in fallback_results_table[
    fallback_results_table["path"].isin(important_paths)
].iterrows():
    print("\n" + row["path"])
    print("-" * 80)
    print(f"Baseline RMSE: {row['baseline_rmse']}")
    print(f"Linear test R²: {row['linear_test_r2']}")
    print(f"Linear RMSE: {row['linear_rmse']}")
    print(f"Random forest test R²: {row['rf_test_r2']}")
    print(f"Random forest RMSE: {row['rf_rmse']}")
    print(f"Best model: {row['best_model']}")
    print(f"Beats baseline: {row['beats_baseline']}")
    print(f"Decision: {row['decision']}")

    print("\nRaw formula:")
    print(row["raw_formula"])

    print("\nStandardized formula:")
    print(row["standardized_formula"])


# ------------------------------------------------------------
# Simple graph: model RMSE comparison
# ------------------------------------------------------------

plot_data = fallback_results_table.copy()

plt.figure(figsize=(10, 6))
plt.barh(plot_data["path"], plot_data["linear_rmse"])
plt.xlabel("Linear model test RMSE")
plt.ylabel("Path")
plt.title("Fallback sleep-quality/stress bridge comparison")
plt.tight_layout()

graph_path = GRAPH_DIR / "fallback_sleep_quality_stress_rmse.png"
plt.savefig(graph_path, dpi=200)
plt.close()

print("\nSaved graph to:")
print(graph_path)

Project folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis

Fallback dataset folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\fallback_sleep_quality_stress

Trying to download dataset automatically from KaggleHub:
uom190346a/sleep-health-and-lifestyle-dataset


100%|██████████| 2.54k/2.54k [00:00<00:00, 1.27MB/s]

Extracting files...

Dataset downloaded to:
C:\Users\luish\.cache\kagglehub\datasets\uom190346a\sleep-health-and-lifestyle-dataset\versions\2

Using dataset file:
C:\Users\luish\.cache\kagglehub\datasets\uom190346a\sleep-health-and-lifestyle-dataset\versions\2\Sleep_health_and_lifestyle_dataset.csv

Dataset shape:
(374, 13)

Columns:
['Person ID', 'Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Blood Pressure', 'Heart Rate', 'Daily Steps', 'Sleep Disorder']


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea



Rows available after cleaning:
374

Descriptive statistics:


...


Fallback model results:


,path,target,predictors,rows_used,baseline_r2,baseline_mae,baseline_rmse,linear_train_r2,linear_test_r2,linear_mae,linear_rmse,rf_train_r2,rf_test_r2,rf_mae,rf_rmse,best_model,beats_baseline,decision,raw_formula,standardized_formula
0,Quality of Sleep → Stress Level,Stress Level,Quality of Sleep,374,-0.0272,1.5851,1.8179,0.7958,0.8375,0.5585,0.7230,0.8535,0.8964,0.4781,0.5772,random forest,Yes,use with caution,Stress Level = 15.136409 -1.336180 * Quality o...,Stress Level = 5.310714 -1.568836 * z(Quality ...
1,Sleep Duration + Quality of Sleep → Stress Level,Stress Level,"Sleep Duration, Quality of Sleep",374,-0.0272,1.5851,1.8179,0.7959,0.8397,0.5566,0.7181,0.9068,0.9350,0.2339,0.4573,random forest,Yes,use with caution,Stress Level = 15.273574 -0.048811 * Sleep Dur...,Stress Level = 5.310714 -0.038307 * z(Sleep Du...
2,Physical Activity Level → Quality of Sleep,Quality of Sleep,Physical Activity Level,374,-0.0169,1.1014,1.2591,0.0275,0.0450,1.0557,1.2202,0.3422,0.2223,0.8908,1.1011,random forest,Yes,use with caution,Quality of Sleep = 6.790611 +0.009350 * Physic...,Quality of Sleep = 7.353571 +0.194556 * z(Phys...
3,Physical Activity Level + Daily Steps → Qualit...,Quality of Sleep,"Physical Activity Level, Daily Steps",374,-0.0169,1.1014,1.2591,0.1027,-0.0095,1.0651,1.2546,0.6194,0.4574,0.6542,0.9197,random forest,Yes,use with caution,Quality of Sleep = 7.834944 +0.027298 * Physic...,Quality of Sleep = 7.353571 +0.568003 * z(Phys...
4,Physical Activity Level → Stress Level,Stress Level,Physical Activity Level,374,-0.0272,1.5851,1.8179,0.0002,-0.0327,1.5897,1.8228,0.1936,0.1617,1.4525,1.6423,random forest,Yes,use with caution,Stress Level = 5.236226 +0.001237 * Physical A...,Stress Level = 5.310714 +0.025743 * z(Physical...
5,Quality of Sleep + Physical Activity Level → S...,Stress Level,"Quality of Sleep, Physical Activity Level",374,-0.0272,1.5851,1.8179,0.8230,0.8355,0.5447,0.7275,0.9115,0.9435,0.2210,0.4264,random forest,Yes,use with caution,Stress Level = 14.591243 -1.377640 * Quality o...,Stress Level = 5.310714 -1.617515 * z(Quality ...



Fallback coefficients:


,path,term,raw_coefficient,standardized_coefficient,training_mean,training_sd
0,Quality of Sleep → Stress Level,Intercept,15.136409,5.310714,,
1,Quality of Sleep → Stress Level,Quality of Sleep,-1.336180,-1.568836,7.353571,1.17412
2,Sleep Duration + Quality of Sleep → Stress Level,Intercept,15.273574,5.310714,,
3,Sleep Duration + Quality of Sleep → Stress Level,Sleep Duration,-0.048811,-0.038307,7.149643,0.784811
4,Sleep Duration + Quality of Sleep → Stress Level,Quality of Sleep,-1.307376,-1.535017,7.353571,1.17412
5,Physical Activity Level → Quality of Sleep,Intercept,6.790611,7.353571,,
6,Physical Activity Level → Quality of Sleep,Physical Activity Level,0.009350,0.194556,60.207143,20.807278
7,Physical Activity Level + Daily Steps → Qualit...,Intercept,7.834944,7.353571,,
8,Physical Activity Level + Daily Steps → Qualit...,Physical Activity Level,0.027298,0.568003,60.207143,20.807278
9,Physical Activity Level + Daily Steps → Qualit...,Daily Steps,-0.000309,-0.493204,6882.142857,1597.375877



Saved fallback model results to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\fallback_sleep_quality_stress_results.csv

Saved fallback coefficients to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\fallback_sleep_quality_stress_coefficients.csv

IMPORTANT FALLBACK FORMULAS

Quality of Sleep → Stress Level
--------------------------------------------------------------------------------
Baseline RMSE: 1.8179
Linear test R²: 0.8375
Linear RMSE: 0.723
Random forest test R²: 0.8964
Random forest RMSE: 0.5772
Best model: random forest
Beats baseline: Yes
Decision: use with caution

Raw formula:
Stress Level = 15.136409 -1.336180 * Quality of Sleep

Standardized formula:
Stress Level = 5.310714 -1.568836 * z(Quality of Sleep)

Physical Activity Level → Quality of Sleep
--------------------------------------------------------------------------------
Baseline RMSE: 1.2591
Linear test R²: 0.045
Linear RMSE: 1.2202
Random forest test R²: 0.2223
Random forest RMS